# 1. Build the demand

Builds every input the simulations need from the public dataset:

* **Side streets:** tertiary roads that are not part of the main-road network (one lane, reduced capacity).
* **Peak-hour OD matrix** from the JICA 2011 surveys: the 9-zone vehicle OD table (PCU/day) is split over the 51 person-trip zones in proportion to person trips, times a peak-hour factor of 0.10, and loaded at up to four junctions per zone.
* **Scenario variants:** morning-biased demand (tidal lanes test), decentralized destinations, and a uniform cut to the same total (its control).
* **Growth scenarios:** the base matrix grown from 2011 at 4.52%/yr (JICA/DoR 2024 forecast) by x1.20 (4.2 years, for comparison with an earlier version of the model), to 2026 (x1.94) and to the end of 2030 (x2.32), in whole platoons of five vehicles.

In [ ]:
import glob, os, shutil, subprocess, sys, zipfile

def find_project(marker):
    """Copy the project tree that contains `marker` from /kaggle/input to /kaggle/working/sim."""
    hits = sorted(glob.glob(f"/kaggle/input/**/{marker}", recursive=True),
                  key=lambda h: ("build-demand" not in h, len(h)))   # prefer step 1's copy
    if not hits:                                        # dataset delivered as a zip
        for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
            zipfile.ZipFile(z).extractall("/kaggle/working/_unzipped")
        hits = sorted(glob.glob(f"/kaggle/working/_unzipped/**/{marker}", recursive=True), key=len)
    src = hits[0][: -len(marker)].rstrip("/")
    shutil.copytree(src, "/kaggle/working/sim", dirs_exist_ok=True)
    os.chdir("/kaggle/working/sim")
    print("project from", src)

def run(*cmd, tries=1):
    """Run a script, streaming its output; retry resumable drivers if a worker is killed."""
    for attempt in range(tries):
        p = subprocess.Popen([sys.executable, "-u", *cmd], stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            print(line, end="")
        if p.wait() == 0:
            return
        print(f"exit code {p.returncode}, attempt {attempt + 1} of {tries}")
    raise RuntimeError(" ".join(cmd))

find_project("scripts/build_od_jica.py")

In [ ]:
run("scripts/build_minor_ways.py")
run("scripts/build_od_jica.py", "--data", "data/jica_net")
for v in ("am", "decentralized", "uniform"):
    run("scripts/build_od_jica.py", "--data", "data/jica_net", "--variant", v)
run("scripts/build_growth_od.py")

In [ ]:
import csv
for f in sorted(glob.glob("data/jica_net/od_matrix*.csv")):
    rows = list(csv.DictReader(open(f)))
    print(f"{os.path.basename(f):<30} {len(rows):>6} pairs {sum(int(r['demand']) for r in rows):>9,} PCU/h")